0) **Imports**

In [62]:
import os, sys
sys.path.append("/home/frocha/sources") # parent folder of ddfenicsx

from timeit import default_timer as timer
import numpy as np

from petsc4py import PETSc
from mpi4py import MPI
from petsc4py.PETSc import ScalarType  # type: ignore

import dolfinx, ufl
print(f"DOLFINx version: {dolfinx.__version__}")
print(f"UFL version: {ufl.__version__}")

from dolfinx import fem,mesh,plot
import dolfinx.fem.petsc
import ddfenicsx as dd
from ddfenicsx.utils.fetricks import symgrad_mandel

import pyvista as pv
from dolfinx.plot import vtk_mesh

DOLFINx version: 0.10.0
UFL version: 2025.2.0


1) **Consititutive behaviour Definition**

Strain Energy
$$
\psi(\varepsilon(u)) = \frac{\lambda}{2}
( tr {\varepsilon}^2 + \frac{\alpha}{2} tr {\varepsilon}^4) + 
\mu ( |\varepsilon|^2  + \frac{\alpha}{2} |\varepsilon|^4).
$$

In [63]:
nu = 0.3 
E = 100.0 
alpha = 10e4
lamb = nu*E/(1.0+nu)/(1.0-2*nu)
mu = E/(1.0+nu)

# alpha : Equivalent to sig = lamb*df.div(u)*df.Identity(2) + mu*(df.grad(u) + df.grad(u).T)
def psi_e(e):
    tr_e = ufl.tr(e)
    e2 = ufl.inner(e,e)
    
    return (0.5*lamb*(tr_e**2 + 0.5*alpha*tr_e**4) +
           mu*(e2 + 0.5*alpha*e2**2))

psi = lambda w: psi_e(ufl.sym(ufl.grad(w)))

2) **Mesh**  

In [64]:
Nx =  50 
Ny =  10 
Lx = 2.0
Ly = 0.5
domain = mesh.create_rectangle(MPI.COMM_WORLD,[[0.0, 0.0],[Lx,Ly]],[Nx,Ny], mesh.CellType.triangle, diagonal = mesh.DiagonalType.right)
tdim = domain.topology.dim
dx = ufl.Measure('dx', domain)

topology, cell_types, geometry = vtk_mesh(domain)
grid = pv.UnstructuredGrid(topology, cell_types, geometry)

# Plot
# plotter = pv.Plotter()
# plotter.add_mesh(grid, show_edges=True)
# plotter.view_xy()
# plotter.enable_parallel_projection()
# plotter.show()

3) **Mesh regions** 

In [65]:
def clamped_boundary(x):
    return np.isclose(x[0],0.0)

fdim = domain.topology.dim-1
clamped_facets = mesh.locate_entities_boundary(domain,fdim,clamped_boundary)

# Neumann boundary conditions
def traction_boundary(x):
    return np.isclose(x[0],Lx)

traction_facets = mesh.locate_entities(domain,fdim,traction_boundary)
traction_facets_tag = mesh.meshtags(domain,fdim,traction_facets,np.full_like(traction_facets,1))
ds = ufl.Measure("ds",domain=domain,subdomain_data=traction_facets_tag)

4) **Spaces**

In [66]:
Uh = fem.functionspace(domain,("Lagrange",1,(tdim,)))
uD = np.array([0.0,0.0],dtype=PETSc.ScalarType)
bc = fem.dirichletbc(uD,fem.locate_dofs_topological(Uh,fdim,clamped_facets),Uh)

5. **Variational Formulation**: Minimisation 
\begin{align} 
\min_{u \in U} \left ( J(u):=\int_{\Omega} \psi(u) dx - \Pi_{ext}(u) \right) \\
F(u; v) = \delta J(u;v) = 0 \quad \forall v \in V , \\
\delta F(u, du; v) = \delta^2 J(u, du;v) \quad \forall v \in V ,
\end{align} 
<br>


In [68]:
du = ufl.TrialFunction(Uh)            # Incremental displacement
v  = ufl.TestFunction(Uh)             # Test function
uh  = fem.Function(Uh)                 # Displacement from previous iteration

ty = -0.1
traction = np.array([0.0, ty])
T = fem.Constant(domain, traction)

P_ext = ufl.inner(T,uh)*ds(1)
P_int = psi(uh)*dx

J = P_int - P_ext

F = ufl.derivative(J, uh, v)
DF = ufl.derivative(F, uh, du) # Optional

6) **Solving**

In [71]:
# Compute solution
start = timer()
petsc_options = {
    "snes_type": "newtonls",
    "snes_linesearch_type": "none",
    "snes_monitor": None,
    "snes_atol": 1e-9,
    "snes_rtol": 1e-9,
    "snes_stol": 0.0,
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps"
}

problem = fem.petsc.NonlinearProblem(F, uh, bcs=[bc], J = DF,  petsc_options = petsc_options, 
                                     petsc_options_prefix='nonlinear_elasticity')

problem.solve()

end = timer()

print("Time spent: ", end - start)
print("Norm L2: ", fem.assemble_scalar(fem.form(ufl.inner(uh,uh)*dx)))

print("Norm l2: ", np.linalg.norm(uh.x.array) )
# 0.5066642165089559

Error: error code 86
[0] SNESSetFromOptions() at /home/conda/feedstock_root/build_artifacts/bld/rattler-build_petsc_1764488351/work/src/snes/interface/snes.c:1181
[0] SNESLineSearchSetFromOptions() at /home/conda/feedstock_root/build_artifacts/bld/rattler-build_petsc_1764488351/work/src/snes/linesearch/interface/linesearch.c:830
[0] SNESLineSearchSetType() at /home/conda/feedstock_root/build_artifacts/bld/rattler-build_petsc_1764488351/work/src/snes/linesearch/interface/linesearch.c:978
[0] Unknown type. Check for miss-spelling or missing package: https://petsc.org/release/install/install/#external-packages
[0] Unable to find requested Line Search type 

7. **Pos-Processing**

In [ ]:
import fetricks.fenics.postprocessing.wrapper_io as iofe
from fetricks import symgrad_mandel, tensor2mandel

def sigma_law(w):    
    e = 0.5*(df.grad(w) + df.grad(w).T) 
    e = df.variable(e)
    return df.diff(psi_e(e),e)

Sh = df.VectorFunctionSpace(mesh, "DG", 0, dim = 3) 
epsh = df.project( symgrad_mandel(uh), Sh)
sig = sigma_law(uh)
sigh = df.project( tensor2mandel(sig), Sh)

uh.rename("u",'')
epsh.rename("eps",'')
sigh.rename("sig",'')

iofe.exportXDMF_gen("bar_nonlinear_vtk.xdmf", fields={'vertex': [uh], 'cell_vector': [epsh, sigh] })
iofe.exportXDMF_checkpoint_gen("bar_nonlinear_sol.xdmf", fields={'vertex': [uh], 'cell': [epsh, sigh]})

Calling FFC just-in-time (JIT) compiler, this may take some time.


8. **Sanity Check**

In [ ]:
# Check for alpha = 0.0 for the linear case
sol_files =  [df.XDMFFile("bar_nonlinear_sol.xdmf"), df.XDMFFile("../linear/bar_sol.xdmf")]

u = []; eps = []; sig = []

for file in sol_files:
    u.append(df.Function(Uh))
    eps.append(df.Function(Sh))
    sig.append(df.Function(Sh))
    
    file.read_checkpoint(u[-1],"u") 
    file.read_checkpoint(eps[-1],"eps")
    file.read_checkpoint(sig[-1],"sig")


print('error L2 disp', np.sqrt(df.assemble( df.inner(u[1] - u[0], u[1] - u[0])*dx) ))
print('error L2 eps', np.sqrt(df.assemble( df.inner(eps[1] - eps[0], eps[1] - eps[0])*dx) ) )
print('error L2 sig', np.sqrt(df.assemble( df.inner(sig[1] - sig[0], sig[1] - sig[0])*dx) ) )

assert np.allclose(u[0].vector().get_local()[:], u[1].vector().get_local()[:])
assert np.allclose(eps[0].vector().get_local()[:], eps[1].vector().get_local()[:])
assert np.allclose(sig[0].vector().get_local()[:], sig[1].vector().get_local()[:]) 

error L2 disp 0.03765547166968704
error L2 eps 0.03444392752955096
error L2 sig 0.24035257432540347


AssertionError: 